<a href="https://colab.research.google.com/github/N-Uma/Machine-Learning-and-Big-Data/blob/main/Feature_Engineering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [9]:
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import DoubleType
from pyspark.ml import Pipeline, Transformer
from pyspark.ml.feature import (
    VectorAssembler,
    StandardScaler,
    OneHotEncoder,
    StringIndexer
)
from pyspark.ml.util import DefaultParamsReadable, DefaultParamsWritable

path = "/content/drive/MyDrive/NYC_Taxi_Yellow_2012"
BASE_PATH = path
PROCESSED_PATH = f"{path}/data/processed/"
INPUT_PATH = PROCESSED_PATH
OUTPUT_PATH = f"{BASE_PATH}/data/final"
MODEL_PATH = f"{BASE_PATH}/models/feature_pipeline_2012"


spark = (
    SparkSession.builder
    .appName("NYCTaxi_FeatureEngineering_2012")
    .config("spark.sql.parquet.enableVectorizedReader", "true")
    .getOrCreate()
)


df = spark.read.parquet(INPUT_PATH)



df_features = (
    df
    .withColumn("pickup_hour", F.hour("tpep_pickup_datetime"))
    .withColumn("day_of_week", F.dayofweek("tpep_pickup_datetime"))
    .withColumn(
        "is_peak_hour",
        F.when(
            F.col("pickup_hour").between(7, 9) |
            F.col("pickup_hour").between(16, 19),
            1
        ).otherwise(0)
    )
    .withColumn(
        "is_weekend",
        F.when(F.col("day_of_week").isin(1, 7), 1).otherwise(0)
    )

    .fillna(0)
)



dow_indexer = StringIndexer(
    inputCol="day_of_week",
    outputCol="dow_index",
    handleInvalid="keep"
)

dow_encoder = OneHotEncoder(
    inputCol="dow_index",
    outputCol="dow_vector"
)

numeric_features = [
    "passenger_count",
    "trip_distance",
    "congestion_surcharge",
    "pickup_hour",
    "is_peak_hour",
    "is_weekend"
]

assembler = VectorAssembler(
    inputCols=numeric_features + ["dow_vector"],
    outputCol="features_unscaled"
)

scaler = StandardScaler(
    inputCol="features_unscaled",
    outputCol="features",
    withStd=True,
    withMean=True
)

pipeline = Pipeline(
    stages=[
        dow_indexer,
        dow_encoder,
        assembler,
        scaler
    ]
)


pipeline_model = pipeline.fit(df_features)
df_final_features = pipeline_model.transform(df_features)


train_df = df_final_features.filter(F.col("pickup_month") <= "2012-10")
val_df   = df_final_features.filter(F.col("pickup_month") == "2012-11")
test_df  = df_final_features.filter(F.col("pickup_month") == "2012-12")


def save_dataset(df, name):
    path = f"{OUTPUT_PATH}/{name}"
    (
        df
        .select("features", "fare_amount")
        .write
        .mode("overwrite")
        .parquet(path)
    )
    print(f" Saved {name} to {path}")

save_dataset(train_df, "train_2012")
save_dataset(val_df, "val_2012")
save_dataset(test_df, "test_2012")


pipeline_model.write().overwrite().save(MODEL_PATH)

print("\n--- Feature Engineering Complete ---")
df_final_features.select("features", "fare_amount").show(5, truncate=False)


 Saved train_2012 to /content/drive/MyDrive/NYC_Taxi_Yellow_2012/data/final/train_2012
 Saved val_2012 to /content/drive/MyDrive/NYC_Taxi_Yellow_2012/data/final/val_2012
 Saved test_2012 to /content/drive/MyDrive/NYC_Taxi_Yellow_2012/data/final/test_2012

--- Feature Engineering Complete ---
+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-----------+
|features                                                                                                                                                                                                                                                                |fare_amount|
+----------------------------------------------------------------------------------------------------------------------------------------------------